# SPY 0DTE Candle-Pattern Strategy

**Honest result from prior run:** bearish_engulfing has real edge (69% WR over 117 trades, +$5,161). Other 9 patterns are net negative or too small to bet on. This run lets you compare:
- All 10 patterns vs only bearish_engulfing
- Asymmetric 10% PT / 20% SL vs symmetric 10/10

## 1. Setup

In [ ]:
import os, sys
REPO = '/content/iron-condor'
BRANCH = 'claude/spy-options-trading-bot-ri4Ms'
if not os.path.isdir(REPO):
    !git clone --quiet -b {BRANCH} https://github.com/coolin815/iron-condor.git {REPO}
else:
    !cd {REPO} && git pull --quiet
SRC = REPO + '/src'
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(REPO)
print('OK')

## 2. Paste your Polygon key

In [ ]:
import os, getpass
os.environ['POLYGON_API_KEY'] = getpass.getpass('Polygon API key: ')
print('Key loaded:', len(os.environ['POLYGON_API_KEY']), 'chars')

## 3a. All 10 patterns, asymmetric 10/20 exits — baseline rerun

In [ ]:
!python -m iron_condor.cli --days 365 --sweep --pt 0.10 --sl 0.20

In [ ]:
import shutil
shutil.copy('results/sweep_trades.csv', 'results/all10_10x20_trades.csv')
shutil.copy('results/sweep_summary.csv', 'results/all10_10x20_summary.csv')
print('Saved baseline to results/all10_10x20_*.csv')

## 3b. All 10 patterns, symmetric 10/10 exits
Break-even WR drops from 67% to 50% — many more patterns become viable.

In [ ]:
!python -m iron_condor.cli --days 365 --sweep --pt 0.10 --sl 0.10

In [ ]:
shutil.copy('results/sweep_trades.csv', 'results/all10_10x10_trades.csv')
shutil.copy('results/sweep_summary.csv', 'results/all10_10x10_summary.csv')
print('Saved symmetric-exits run to results/all10_10x10_*.csv')

## 3c. ONLY bearish_engulfing (the proven winner), asymmetric 10/20

In [ ]:
!python -m iron_condor.cli --days 365 --sweep --pt 0.10 --sl 0.20 --pattern bearish_engulfing

In [ ]:
shutil.copy('results/sweep_trades.csv', 'results/be_only_trades.csv')
shutil.copy('results/sweep_summary.csv', 'results/be_only_summary.csv')
print('Saved bearish_engulfing-only run to results/be_only_*.csv')

## 4. Side-by-side: which scenario won?

In [ ]:
import pandas as pd
for label, path in [
    ('A) All 10 patterns, 10/20 exits',          'results/all10_10x20_summary.csv'),
    ('B) All 10 patterns, 10/10 symmetric',      'results/all10_10x10_summary.csv'),
    ('C) bearish_engulfing only, 10/20',         'results/be_only_summary.csv'),
]:
    s = pd.read_csv(path)
    best = s.iloc[0]
    print(f'\n=== {label} ===')
    print(f"  best config:   {best['config']}")
    print(f"  return:        {best['total_return_pct']:>+8.1f}%")
    print(f"  win rate:      {best['win_rate']:>8.1%}")
    print(f"  trades:        {best['n_trades']}")
    print(f"  max drawdown:  {best['max_drawdown_pct']:>+8.1f}%")
    print(f"  avg net P&L:   ${best['avg_net_pnl']:>+8.2f}")

## 5. Per-pattern breakdown for the all-10 / 10x10 run

In [ ]:
trades = pd.read_csv('results/all10_10x10_trades.csv')
taken = trades[trades['exit_reason'].isin(['profit', 'stop', 'time_stop'])]
if not taken.empty:
    print('=== By pattern (10/10 exits) ===')
    print(taken.groupby('pattern').agg(
        trades=('net_pnl', 'count'),
        win_rate=('net_pnl', lambda s: (s > 0).mean()),
        avg_pnl=('net_pnl', 'mean'),
        median_pnl=('net_pnl', 'median'),
        total_pnl=('net_pnl', 'sum'),
    ).round(2).sort_values('total_pnl', ascending=False))

## 6. Equity curve — best of each scenario

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 5))
for label, summ_path, trades_path in [
    ('A: all/10x20', 'results/all10_10x20_summary.csv', 'results/all10_10x20_trades.csv'),
    ('B: all/10x10', 'results/all10_10x10_summary.csv', 'results/all10_10x10_trades.csv'),
    ('C: be-only/10x20', 'results/be_only_summary.csv', 'results/be_only_trades.csv'),
]:
    summ = pd.read_csv(summ_path)
    cfg = summ.iloc[0]['config']
    tr = pd.read_csv(trades_path)
    sub = tr[tr['config'] == cfg].sort_values('day')
    ax.plot(pd.to_datetime(sub['day']), sub['balance_after'], label=label, linewidth=1.5)
ax.axhline(1500, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Date'); ax.set_ylabel('Balance ($)')
ax.set_title('Equity curves: best config from each scenario')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()